In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df=pd.read_csv('novagen_dataset.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9549 entries, 0 to 9548
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    9549 non-null   float64
 1   BMI                    9549 non-null   float64
 2   Blood_Pressure         9549 non-null   float64
 3   Cholesterol            9549 non-null   float64
 4   Glucose_Level          9549 non-null   float64
 5   Heart_Rate             9549 non-null   float64
 6   Sleep_Hours            9549 non-null   float64
 7   Exercise_Hours         9549 non-null   float64
 8   Water_Intake           9549 non-null   float64
 9   Stress_Level           9549 non-null   float64
 10  Target                 9549 non-null   int64  
 11  Smoking                9549 non-null   int64  
 12  Alcohol                9549 non-null   int64  
 13  Diet                   9549 non-null   int64  
 14  MentalHealth           9549 non-null   int64  
 15  Phys

In [4]:
cat_data=df.select_dtypes(include=['bool']).columns

In [5]:
cat_data

Index(['Diet_Type__Vegan', 'Diet_Type__Vegetarian', 'Blood_Group_AB',
       'Blood_Group_B', 'Blood_Group_O'],
      dtype='object')

In [6]:
df=pd.get_dummies(df,columns=cat_data,drop_first=True,dtype=int)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9549 entries, 0 to 9548
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         9549 non-null   float64
 1   BMI                         9549 non-null   float64
 2   Blood_Pressure              9549 non-null   float64
 3   Cholesterol                 9549 non-null   float64
 4   Glucose_Level               9549 non-null   float64
 5   Heart_Rate                  9549 non-null   float64
 6   Sleep_Hours                 9549 non-null   float64
 7   Exercise_Hours              9549 non-null   float64
 8   Water_Intake                9549 non-null   float64
 9   Stress_Level                9549 non-null   float64
 10  Target                      9549 non-null   int64  
 11  Smoking                     9549 non-null   int64  
 12  Alcohol                     9549 non-null   int64  
 13  Diet                        9549 

In [26]:
X=df.drop(columns='Target')
y=df['Target']

In [27]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,stratify=y,random_state=42
)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
from sklearn.tree import DecisionTreeClassifier

In [30]:
model =DecisionTreeClassifier()
model.fit(X_train_scaled,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [31]:
from sklearn.metrics import accuracy_score
y_pred=model.predict(X_test_scaled)
y_train_pred=model.predict(X_train_scaled)
print('train accuracy : ',accuracy_score(y_train,y_train_pred))
print('test accuracy : ',accuracy_score(y_test,y_pred))

train accuracy :  1.0
test accuracy :  0.8958115183246074


In [33]:
# Check for logistic

from sklearn.linear_model import LogisticRegression
lr=LogisticRegression(
    penalty='l2',
    max_iter=5000
)
lr.fit(X_train_scaled,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [34]:
y_pred=lr.predict(X_test_scaled)
y_train_pred=lr.predict(X_train_scaled)
print('train accuracy : ',accuracy_score(y_train,y_train_pred))
print('test accuracy : ',accuracy_score(y_test,y_pred))

train accuracy :  0.8175153815944496
test accuracy :  0.8136125654450261


In [36]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="euclidean"
)

knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))

KNN Accuracy: 0.8832460732984293


In [37]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(
    n_estimators=601,
    oob_score=True,
    max_depth=None
)
rf.fit(X_train_scaled,y_train)
y_pred=rf.predict(X_test_scaled)
print('OOB Score : ',rf.oob_score_)
print('accuracy : ',accuracy_score(y_test,y_pred))

OOB Score :  0.9388663437622725
accuracy :  0.9403141361256544


In [38]:
from sklearn.ensemble import VotingClassifier
lr=LogisticRegression(max_iter=5000)
vc=VotingClassifier(
    estimators=[
        ('lr',lr),
        ('rf',rf),
        ('knn',knn)
    ]
)

In [39]:
vc.fit(X_train_scaled,y_train)

,estimators,"[('lr', ...), ('rf', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [40]:
y_pred=vc.predict(X_test_scaled)
print('accuracy : ',accuracy_score(y_test,y_pred))

accuracy :  0.9261780104712042
